Load the silver data


In [3]:
from pathlib import Path
import pandas as pd
import plotly.express as px

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
train = pd.read_parquet(ROOT / "data/silver/cmapss_train_FD001.parquet")
sensors = [c for c in train.columns if c.startswith(("op_", "s_"))]
train.shape

(20631, 27)

How long do engines live?

In [4]:
life = train.groupby("unit")["cycle"].max()
print(life.describe())
px.histogram(life, nbins=30, title="Engine lifetime (cycles to failure)")

count    100.000000
mean     206.310000
std       46.342749
min      128.000000
25%      177.000000
50%      199.000000
75%      229.250000
max      362.000000
Name: cycle, dtype: float64


Which sensors actually change?

In [5]:
spread = train[sensors].std().sort_values()
spread.to_frame("std")

,std
s_1,0.000000e+00
op_3,0.000000e+00
s_10,0.000000e+00
s_19,0.000000e+00
s_18,0.000000e+00
s_16,3.469531e-18
s_5,5.329200e-15
op_2,2.930621e-04
s_6,1.388985e-03
op_1,2.187313e-03


Which sensors track degradation?

In [6]:
corr = train[sensors + ["rul"]].corr()["rul"].drop("rul").sort_values()
px.bar(corr, title="Correlation of each sensor with RUL")

Watch sensors drift toward failure

In [7]:
sample = train[train["unit"].isin([1, 2, 3, 4, 5])]
for s in ["s_2", "s_4", "s_11", "s_12"]:
    fig = px.line(sample, x="rul", y=s, color="unit", title=f"{s} vs RUL")
    fig.update_xaxes(autorange="reversed")  # failure at the right edge
    fig.show()
    

FD001 EDA findings

1.Engine life varies widely (128–362 cycles, median 199), so age-based maintenance wastes about a third of typical life. Sensor-based prediction is justified.

2.Seven sensors are constant under FD001's single operating condition and are dropped. Fourteen informative sensors are kept.

3.Temperatures and pressures rise toward failure while coolant bleeds fall, consistent with compressor efficiency loss.

4.Degradation is two-phase (plateau, then wear-out from RUL ≈ 125), so the RUL 5.target is capped at 125.

5.Sensors are noisy relative to the trend, so rolling-window features are needed.

6.Next: check sensor-to-sensor redundancy and investigate s_9 and s_14 per engine.

In [8]:
sample = train[train["unit"].isin([1, 2, 3, 4, 5])]
for s in ["s_9", "s_14"]:
    fig = px.line(sample, x="rul", y=s, color="unit", title=f"{s} vs RUL")
    fig.update_xaxes(autorange="reversed")  # failure at the right edge
    fig.show()

In [9]:
def drift(g, col, n=20):
    g = g.sort_values("cycle")
    return g[col].tail(n).mean() - g[col].head(n).mean()

for col in ["s_9", "s_14", "s_4", "s_12"]:
    d = train.groupby("unit").apply(drift, col=col)
    print(f"{col}: up={int((d > 0).sum())}, down={int((d < 0).sum())}")

s_9: up=71, down=29
s_14: up=57, down=43
s_4: up=100, down=0
s_12: up=0, down=100


### Direction of sensor drift toward failure

**Observation**

Plots (5 engines):
- s_4 and s_11 rise steadily and s_12 falls steadily as RUL approaches 0, in the same direction for every engine.
- s_9 and s_14 stay flat for most of the engine's life, then diverge near failure: units 3 and 4 rise sharply in the last ~60 cycles, units 2 and 5 rise gently, and unit 1 falls.
- Unit 5 starts at a higher healthy baseline (~9,065 vs ~9,050 for s_9), showing engine-to-engine variation even when healthy.

Drift check across all 100 engines (mean of last 20 cycles minus mean of first 20 cycles):

| Sensor | Engines up | Engines down |
|---|---|---|
| s_4  | 100 | 0   |
| s_12 | 0   | 100 |
| s_9  | 71  | 29  |
| s_14 | 57  | 43  |

**Interpretation**

- s_4 and s_12 are **universal degradation indicators**: every engine moves the same way as it wears.
- s_9 and s_14 (core speed) are **engine-specific indicators**: they move away from their healthy value near failure, but the direction depends on how each engine wears. Core speed reflects the control system compensating to maintain thrust, so it can shift either way.
- This explains their weak Pearson correlation with RUL (−0.39 for s_9): engines drifting in opposite directions partly cancel out when pooled. The weak correlation hides a real signal rather than indicating no signal.
- s_14 is more evenly split than s_9. Some engines show very small drift, where noise likely decides the sign, so direction alone is unreliable for these sensors.

**Decision**

- Keep s_9 and s_14. Do not drop sensors based on correlation alone.
- Add **baseline-deviation features** (reading minus each engine's own first-20-cycle mean) and their **absolute values**, which grow toward failure regardless of direction.
- Add **rolling-slope features** to capture the sharp late-stage acceleration seen in units 3 and 4.
- Weak-but-non-constant sensors (e.g., s_6) are kept for now; feature selection in the optimization phase will decide whether they add value.

In [10]:
from assetmind.features.cmapss import CmapssFeatureBuilder
fb = CmapssFeatureBuilder().fit(train)
print("Dropped:", fb.dropped_)

Dropped: ['s_1', 's_10', 's_16', 's_18', 's_19', 's_5']


### Check that the new features did their job

We built absdev to turn the split vote of s_9 and s_14 into a consistent one. Let's verify that rather than assume it

In [11]:
feats = fb.transform(train)
for col in ["s_9", "s_9_absdev", "s_14", "s_14_absdev"]:
    d = feats.groupby("unit").apply(drift, col=col)
    print(f"{col:12s} up={int((d > 0).sum()):3d}  down={int((d < 0).sum()):3d}")

s_9          up= 71  down= 29
s_9_absdev   up= 99  down=  1
s_14         up= 57  down= 43
s_14_absdev  up= 97  down=  3


### Verifying the baseline-deviation feature

Absolute deviation from each engine's own healthy baseline turns direction-dependent sensors into consistent degradation indicators:

| Feature | Engines up | Engines down |
|---|---|---|
| s_9 (raw) | 71 | 29 |
| s_9_absdev | 99 | 1 |
| s_14 (raw) | 57 | 43 |
| s_14_absdev | 97 | 3 |

Core-speed sensors drift away from their healthy value in either direction; measuring distance from baseline instead of direction recovers a signal that nearly every engine shares.